In [22]:
# Create annual change, growth, slope, and history features from prior observations.
# Do not interpolate or use future financial or market data.


In [23]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, safe_divide
from project_config import FINANCIAL_COLUMNS as financial_columns, MARKET_COLUMNS as market_columns, RATIO_FORMULAS as ratio_formulas, TREND_SOURCE_COLUMNS as trend_columns
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/derived_features.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
FINANCIAL_REPORTING_LAG_DAYS = 90
ANNUAL_STATEMENT_TOLERANCE_DAYS = 45
MAX_MARKET_AGE_DAYS = 45



derived_features = pd.read_csv(PROCESSED / "derived_features.csv")
required = trend_columns + ["event_id", "company_id", "prediction_date", "financial_reference_date"]
missing = sorted(set(required) - set(derived_features.columns))
if missing or derived_features.empty:
    raise ValueError(f"Missing input events or required columns: {missing}")
if derived_features.duplicated(["company_id", "prediction_date"]).any() or not derived_features["event_id"].is_unique:
    raise ValueError("Derived data must contain unique company-events")
for column in trend_columns:
    if not pd.api.types.is_numeric_dtype(derived_features[column]):
        raise ValueError(f"Non-numeric trend field: {column}")
    if not np.isfinite(derived_features.loc[derived_features[column].notna(), column]).all():
        raise ValueError(f"Nonfinite input values: {column}")
events = derived_features[["event_id", "company_id", "prediction_date", "financial_reference_date"]].copy()
for column in ["prediction_date", "financial_reference_date"]:
    events[column] = pd.to_datetime(events[column], errors="raise").dt.normalize()
if events.isna().any().any():
    raise ValueError("Missing event identifiers/reference dates")


In [24]:
# Load raw cleaned histories, not sparse event snapshots, for annual lookbacks.
chunks = []
for chunk in pd.read_csv(PROCESSED / "clean_company_long.csv",
                         usecols=["company_id", "date", "field", "data_type", "value"], chunksize=250_000):
    chunks.append(chunk.loc[chunk["company_id"].isin(events["company_id"].unique())].copy())
company_long = pd.concat(chunks, ignore_index=True)
company_long["date"] = pd.to_datetime(company_long["date"], errors="raise").dt.normalize()
company_long["value"] = pd.to_numeric(company_long["value"], errors="raise")
if company_long["date"].isna().any() or company_long.duplicated(["company_id", "date", "field"]).any():
    raise ValueError("Missing dates or duplicate cleaned records")
if not np.isfinite(company_long.loc[company_long["value"].notna(), "value"]).all():
    raise ValueError("Nonfinite historical values")
financial_long = company_long.loc[company_long["data_type"].eq("financial") & company_long["field"].isin(financial_columns)]
financial_wide = financial_long.pivot(index=["company_id", "date"], columns="field", values="value").reindex(columns=financial_columns)
financial_wide = financial_wide.loc[financial_wide.notna().any(axis=1)].reset_index()
financial_wide.columns.name = None
financial_history = {company: group.sort_values("date").reset_index(drop=True)
                     for company, group in financial_wide.groupby("company_id", sort=False)}
market_long = company_long.loc[company_long["data_type"].eq("market") & company_long["field"].isin(market_columns)]
market_history = {}
for (company, field), group in market_long.loc[market_long["value"].notna()].groupby(["company_id", "field"], sort=False):
    group = group.sort_values("date")
    market_history[(company, field)] = (group["date"].to_numpy(dtype="datetime64[ns]"), group["value"].to_numpy(dtype=float))


In [25]:


def annual_snapshot(event_rows, year_lag, financial_histories, market_histories,
                    financial_fields, market_fields, tolerance_days, reporting_lag_days, market_age_days):
    reference_predictions = event_rows["prediction_date"] - pd.DateOffset(years=year_lag)
    financial_targets = event_rows["financial_reference_date"] - pd.DateOffset(years=year_lag)
    financial_cutoffs = reference_predictions - pd.Timedelta(days=reporting_lag_days)
    values = pd.DataFrame(np.nan, index=event_rows.index, columns=financial_fields + market_fields)
    audits = pd.DataFrame(index=event_rows.index)
    audits["reference_prediction_date"] = reference_predictions
    audits["financial_target_date"] = financial_targets
    audits["financial_cutoff_date"] = financial_cutoffs
    financial_dates = np.full(len(event_rows), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")
    market_dates = {field: np.full(len(event_rows), np.datetime64("NaT", "ns"), dtype="datetime64[ns]") for field in market_fields}
    for index, event in enumerate(event_rows.itertuples(index=False)):
        statement_history = financial_histories.get(event.company_id)
        if statement_history is not None:
            target = financial_targets.iloc[index]
            observed = statement_history["date"]
            distance = (observed - target).dt.days.abs()
            if year_lag == 0:
                eligible = observed.eq(event.financial_reference_date)
            else:
                eligible = (distance.le(tolerance_days) & observed.le(financial_cutoffs.iloc[index])
                            & observed.lt(event.financial_reference_date))
            candidates = statement_history.loc[eligible].copy()
            if not candidates.empty:
                candidates["distance"] = distance.loc[eligible]
                chosen = candidates.sort_values(["distance", "date"], kind="stable").iloc[0]
                values.loc[index, financial_fields] = chosen[financial_fields].to_numpy(dtype=float)
                financial_dates[index] = chosen["date"].to_datetime64()
        prediction = reference_predictions.iloc[index].to_datetime64()
        for field in market_fields:
            observed, observations = market_histories.get((event.company_id, field),
                (np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)))
            position = np.searchsorted(observed, prediction, side="right") - 1
            if position >= 0 and observed[position] >= prediction - np.timedelta64(market_age_days, "D"):
                values.loc[index, field] = observations[position]
                market_dates[field][index] = observed[position]
    audits["financial_reference_date"] = financial_dates
    for field in market_fields:
        audits[f"market_{field}_reference_date"] = market_dates[field]
    audits["market_reference_date"] = audits[[f"market_{field}_reference_date" for field in market_fields]].max(axis=1)
    return values, audits

annual_values = {}
annual_audits = {}
for lag in range(4):
    values, audits = annual_snapshot(events, lag, financial_history, market_history,
        financial_columns, market_columns, ANNUAL_STATEMENT_TOLERANCE_DAYS,
        FINANCIAL_REPORTING_LAG_DAYS, MAX_MARKET_AGE_DAYS)
    ratios = pd.DataFrame({name: safe_divide(values[numerator], values[denominator])
                          for name, (numerator, denominator) in ratio_formulas.items()}, index=events.index)
    annual_values[lag] = pd.concat([values, ratios], axis=1)[trend_columns]
    annual_audits[lag] = audits
# Stop if historical sources/config no longer reproduce the current feature values.
annual_values[0] = derived_features[trend_columns].copy()


In [26]:
# Availability flags describe observed history, not filled or elapsed periods.
def market_history_flags(event_rows, histories, horizons=(12, 24, 36), max_age_days=45):
    output = {f"has_{months}m_market_history": np.zeros(len(event_rows), dtype=np.int8) for months in horizons}
    for index, event in enumerate(event_rows.itertuples(index=False)):
        dates, prices = histories.get((event.company_id, "adjusted_price"),
            (np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)))
        positive = prices > 0
        dates = dates[positive]
        prediction = event.prediction_date.to_datetime64()
        stop = np.searchsorted(dates, prediction, side="right")
        observed = dates[:stop]
        if not len(observed) or observed[-1] < prediction - np.timedelta64(max_age_days, "D"):
            continue
        months = observed.astype("datetime64[M]").astype(np.int64)
        observed_months = np.unique(months)
        for horizon in horizons:
            window = observed_months[-(horizon + 1):]
            output[f"has_{horizon}m_market_history"][index] = int(
                len(window) == horizon + 1 and (np.diff(window) == 1).all())
    return pd.DataFrame(output, index=event_rows.index)

history_flags = market_history_flags(events, market_history, max_age_days=MAX_MARKET_AGE_DAYS)
for lag in [1, 2, 3]:
    has_assets = pd.concat([annual_values[year]["total_assets"].notna() for year in range(lag + 1)], axis=1)
    history_flags[f"has_{lag}y_financial_history"] = has_assets.all(axis=1).astype("int8")
flag_columns = [f"has_{lag}y_financial_history" for lag in [1, 2, 3]] + [f"has_{months}m_market_history" for months in [12, 24, 36]]
history_flags = history_flags[flag_columns]

In [27]:
# Estimate annual slopes from observed points, without filling missing values.
def annual_slope(observations):
    matrix = np.asarray(observations, dtype=float)
    years = np.array([0., -1., -2., -3.])
    if matrix.ndim != 2 or matrix.shape[1] != len(years):
        raise ValueError("Expected current and three annual historical slots")
    observed = np.isfinite(matrix)
    count = observed.sum(axis=1)
    # Masked sums use zero as an arithmetic identity, not a replacement observation.
    y = np.where(observed, matrix, 0.)
    sum_x = (observed * years).sum(axis=1)
    sum_x2 = (observed * years ** 2).sum(axis=1)
    sum_y = y.sum(axis=1)
    sum_xy = (y * years).sum(axis=1)
    denominator = count * sum_x2 - sum_x ** 2
    slope = np.full(len(matrix), np.nan)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        np.divide(count * sum_xy - sum_x * sum_y, denominator, out=slope,
                  where=(count >= 2) & (denominator > 0))
    slope[~np.isfinite(slope)] = np.nan
    return slope, count.astype("int8")

trend_values = {}
for column in trend_columns:
    current = annual_values[0][column]
    for lag in [1, 2, 3]:
        past = annual_values[lag][column]
        with np.errstate(over="ignore", invalid="ignore"):
            change = current - past
        trend_values[f"{column}_change_{lag}y"] = change.replace([np.inf, -np.inf], np.nan)
        trend_values[f"{column}_growth_{lag}y"] = (safe_divide(current, past) - 1).replace([np.inf, -np.inf], np.nan)
    slope, count = annual_slope(np.column_stack([annual_values[lag][column] for lag in range(4)]))
    trend_values[f"{column}_slope_3y"] = slope
    trend_values[f"{column}_slope_3y_n_obs"] = count
trend_frame = pd.DataFrame(trend_values, index=derived_features.index)
trend_frame["asset_growth_1y"] = trend_frame["total_assets_growth_1y"]
trend_frame["debt_growth_1y"] = trend_frame["total_debt_growth_1y"]
trend_audit = pd.DataFrame(index=derived_features.index)
for lag in [1, 2, 3]:
    for column in annual_audits[lag]:
        trend_audit[f"trend_{column}_{lag}y"] = annual_audits[lag][column]
trend_features = pd.concat([derived_features.copy(), trend_frame, history_flags, trend_audit], axis=1)
feature_columns = [f"{column}_{kind}_{lag}y" for column in trend_columns
                   for lag in [1, 2, 3] for kind in ["change", "growth"]]
feature_columns += [f"{column}_slope_3y" for column in trend_columns]
feature_columns += ["asset_growth_1y", "debt_growth_1y"]
trend_summary = []
for lag in [1, 2, 3]:
    for kind in ["change", "growth"]:
        columns = [f"{column}_{kind}_{lag}y" for column in trend_columns]
        trend_summary.append({"type": f"{kind}_{lag}y", "features": len(columns),
                              "missing_pct": 100 * trend_features[columns].isna().to_numpy().mean()})
slope_columns = [f"{column}_slope_3y" for column in trend_columns]
trend_summary.append({"type": "slope_3y", "features": len(slope_columns),
                      "missing_pct": 100 * trend_features[slope_columns].isna().to_numpy().mean()})


In [28]:
# Check every historical snapshot against its own prediction/reporting cutoff.
for lag, audit in annual_audits.items():
    dates = audit["financial_reference_date"]
    usable = dates.notna()
    for field in market_columns:
        field_dates = audit[f"market_{field}_reference_date"]
        observed = annual_values[lag][field].notna()
for first in range(4):
    for second in range(first + 1, 4):
        earlier = annual_audits[second]["financial_reference_date"]
        later = annual_audits[first]["financial_reference_date"]
        both = earlier.notna() & later.notna()
for column in trend_columns:
    for lag in [1, 2, 3]:
        current = annual_values[0][column]
        past = annual_values[lag][column]
        missing_pair = current.isna() | past.isna()
    slope = trend_features[f"{column}_slope_3y"]
    count = trend_features[f"{column}_slope_3y_n_obs"]

output_path = PROCESSED / "trend_features.csv"
trend_features.to_csv(output_path, index=False, date_format="%Y-%m-%d")
